# Kuka Anomaly Detection — Multi-seed Statistical Comparison
**Project AM01 — Politecnico di Torino 2026**

This notebook:
1. Mounts your Google Drive
2. Installs dependencies
3. Runs AE vs AAE across 6 seeds
4. Performs Wilcoxon signed-rank test
5. Saves all results back to your Drive

**Before running:** Make sure your Drive has this structure:
```
MyDrive/
  code/
  KukaVelocityDataset/
```

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# !! EDIT THIS if your project folder is not directly in MyDrive !!
PROJECT_ROOT = '/content/drive/MyDrive'

CODE_DIR    = os.path.join(PROJECT_ROOT, 'code')
DATASET_DIR = os.path.join(PROJECT_ROOT, 'KukaVelocityDataset')

assert os.path.isdir(CODE_DIR),    f'Cannot find code/ at {CODE_DIR}'
assert os.path.isdir(DATASET_DIR), f'Cannot find KukaVelocityDataset/ at {DATASET_DIR}'
print('Drive mounted and folders found!')
print('code/    :', CODE_DIR)
print('dataset/ :', DATASET_DIR)

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!pip install -q scipy pyyaml scikit-learn
# torch, numpy, matplotlib are pre-installed in Colab
print('Dependencies ready.')

In [ ]:
# ── Cell 3: Add code/ to Python path ────────────────────────────────────────
import sys
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'scripts'))

# Patch the dataset path so src/utils.py finds the data on Drive
import src.utils as _utils
from pathlib import Path
_utils.dataset_dir = lambda: Path(DATASET_DIR)

# Verify data files are present
for fname in ['KukaNormal.npy', 'KukaSlow.npy', 'KukaColumnNames.npy']:
    fpath = os.path.join(DATASET_DIR, fname)
    assert os.path.exists(fpath), f'Missing: {fpath}'
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname}  ({size_mb:.1f} MB)')
print('All dataset files found!')

In [ ]:
# ── Cell 4: Check GPU ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = torch.device('cuda')
else:
    print('No GPU found — running on CPU (will be slower)')
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
import yaml

cfg_path = os.path.join(CODE_DIR, 'configs', 'default.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# !! EDIT THESE if you want a different run !!────────────────────────
N_EPOCHS = 14        # epochs per model per seed (14 matches original run)
SEEDS    = [42, 7, 13, 99, 2024, 314]  # 6 seeds → Wilcoxon min p = 0.03125
# ──────────────────────────────────────────────────────────────────────

print(f'Epochs per model: {N_EPOCHS}')
print(f'Seeds:            {SEEDS}')
print(f'Total runs:       {len(SEEDS)*2}  (AE + AAE per seed)')
print(f'Config loaded from: {cfg_path}')

In [ ]:
# ── Cell 6: Run all seeds ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import time

from _runner_common import build_data
from src.metrics import compute_metrics
from src.models.ae import AEConfig, ConvAE
from src.models.aae import AAEConfig, AdversarialAE
from src.scoring import threshold_at_fpr
from src.training import collect_anomaly_scores, train_ae, train_aae
from src.utils import seed_everything

OUT_DIR = os.path.join(CODE_DIR, 'results', 'multi_run')
os.makedirs(OUT_DIR, exist_ok=True)

rows = []
total_start = time.time()

for seed in SEEDS:
    print(f'\n{"="*55}')
    print(f'  SEED {seed}  ({SEEDS.index(seed)+1}/{len(SEEDS)})')
    print(f'{"="*55}')
    seed_everything(seed)
    cfg_seed = {**cfg, 'seed': seed}

    ds, splits, loaders = build_data(cfg_seed)

    ae_cfg = AEConfig(
        n_features  = splits.n_features,
        window      = splits.window,
        channels    = tuple(cfg['model']['channels']),
        kernel_size = cfg['model']['kernel_size'],
        latent_dim  = cfg['model']['latent_dim'],
    )

    # ---- AE ----
    t0 = time.time()
    ae_model  = ConvAE(ae_cfg)
    ae_result = train_ae(
        ae_model, loaders.train, loaders.val_normal,
        n_epochs=N_EPOCHS, lr=cfg['train']['lr_ae'],
        weight_decay=cfg['train']['weight_decay'],
        device=DEVICE, log_every=N_EPOCHS,
    )
    val_s_ae, _, _         = collect_anomaly_scores(ae_result.model, loaders.val_normal,   DEVICE)
    tst_s_ae, tst_y, tst_a = collect_anomaly_scores(ae_result.model, loaders.stacked_test, DEVICE)
    thr_ae = threshold_at_fpr(val_s_ae, fpr=cfg['eval']['fpr_target'])
    m_ae   = compute_metrics(tst_s_ae, tst_y, thr_ae.value, actions=tst_a)
    print(f'  AE  ROC={m_ae.roc_auc:.4f}  PR={m_ae.pr_auc:.4f}  '
          f'F1={m_ae.f1_at_thr:.4f}  valMSE={ae_result.val_losses[-1]:.4f}  '
          f'({time.time()-t0:.0f}s)')

    # ---- AAE ----
    t0 = time.time()
    aae_cfg   = AAEConfig(ae=ae_cfg,
                          discriminator_hidden=tuple(cfg['aae']['discriminator_hidden']),
                          adv_weight=cfg['aae']['adv_weight'])
    aae_model  = AdversarialAE(aae_cfg)
    aae_result = train_aae(
        aae_model, loaders.train, loaders.val_normal,
        n_epochs=N_EPOCHS, lr_ae=cfg['train']['lr_ae'],
        lr_disc=cfg['train']['lr_disc'],
        weight_decay=cfg['train']['weight_decay'],
        device=DEVICE, log_every=N_EPOCHS,
    )
    val_s_aae, _, _      = collect_anomaly_scores(aae_result.model, loaders.val_normal,   DEVICE)
    tst_s_aae, _, _      = collect_anomaly_scores(aae_result.model, loaders.stacked_test, DEVICE)
    thr_aae = threshold_at_fpr(val_s_aae, fpr=cfg['eval']['fpr_target'])
    m_aae   = compute_metrics(tst_s_aae, tst_y, thr_aae.value, actions=tst_a)
    print(f'  AAE ROC={m_aae.roc_auc:.4f}  PR={m_aae.pr_auc:.4f}  '
          f'F1={m_aae.f1_at_thr:.4f}  valMSE={aae_result.val_losses[-1]:.4f}  '
          f'({time.time()-t0:.0f}s)')

    rows.append({
        'seed':          seed,
        'ae_roc_auc':    m_ae.roc_auc,
        'ae_pr_auc':     m_ae.pr_auc,
        'ae_f1_fpr':     m_ae.f1_at_thr,
        'ae_val_mse':    ae_result.val_losses[-1],
        'aae_roc_auc':   m_aae.roc_auc,
        'aae_pr_auc':    m_aae.pr_auc,
        'aae_f1_fpr':    m_aae.f1_at_thr,
        'aae_val_mse':   aae_result.val_losses[-1],
        'delta_roc_auc': m_ae.roc_auc   - m_aae.roc_auc,
        'delta_pr_auc':  m_ae.pr_auc    - m_aae.pr_auc,
        'delta_f1_fpr':  m_ae.f1_at_thr - m_aae.f1_at_thr,
    })

    # Save incrementally so a Colab disconnect doesn't lose everything
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, 'results_raw.csv'), index=False)
    print(f'  [checkpoint saved after seed {seed}]')

df = pd.DataFrame(rows)
print(f'\nTotal time: {(time.time()-total_start)/60:.1f} min')
print('\nAll results:')
print(df[['seed','ae_roc_auc','aae_roc_auc','delta_roc_auc',
          'ae_pr_auc','aae_pr_auc','delta_pr_auc']].to_string(index=False))

In [ ]:
# ── Cell 7: Wilcoxon signed-rank test ────────────────────────────────────────
from scipy.stats import wilcoxon

report_lines = []
report_lines.append('=' * 60)
report_lines.append('  WILCOXON SIGNED-RANK TEST  (AE vs AAE, paired by seed)')
report_lines.append(f'  H1: AE > AAE   |   n = {len(df)}')
report_lines.append('=' * 60)

for metric, col_ae, col_aae in [
    ('ROC-AUC',   'ae_roc_auc',  'aae_roc_auc'),
    ('PR-AUC',    'ae_pr_auc',   'aae_pr_auc'),
    ('F1@FPR-5%', 'ae_f1_fpr',   'aae_f1_fpr'),
]:
    ae_v  = df[col_ae].values
    aae_v = df[col_aae].values
    diff  = ae_v - aae_v
    n_win = int((diff > 0).sum())

    stat, p = wilcoxon(ae_v, aae_v, alternative='greater', zero_method='wilcox')
    sig = '*** SIGNIFICANT (p < 0.05)' if p < 0.05 else \
          '~ marginal (p < 0.10)'      if p < 0.10 else \
          'not significant'

    report_lines.append(f'\n{metric}')
    report_lines.append(f'  AE  mean={ae_v.mean():.4f}  std={ae_v.std():.4f}')
    report_lines.append(f'  AAE mean={aae_v.mean():.4f}  std={aae_v.std():.4f}')
    report_lines.append(f'  AE wins {n_win}/{len(df)} seeds')
    report_lines.append(f'  Wilcoxon W={stat:.1f}  p={p:.4f}  -> {sig}')

report_lines.append('\n' + '=' * 60)
report_lines.append('Note: minimum achievable p with n=6 is 0.03125.')
report_lines.append('=' * 60)

report = '\n'.join(report_lines)
print(report)

report_path = os.path.join(OUT_DIR, 'wilcoxon_report.txt')
with open(report_path, 'w') as f:
    f.write(report)
print(f'\nReport saved to Drive: {report_path}')

In [ ]:
# ── Cell 8: Summary plot ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

metrics_plot = [
    ('ROC-AUC',   'ae_roc_auc',  'aae_roc_auc'),
    ('PR-AUC',    'ae_pr_auc',   'aae_pr_auc'),
    ('F1@FPR-5%', 'ae_f1_fpr',   'aae_f1_fpr'),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, (title, col_ae, col_aae) in zip(axes, metrics_plot):
    ae_v  = df[col_ae].values
    aae_v = df[col_aae].values

    bp = ax.boxplot([ae_v, aae_v], labels=['AE', 'AAE'],
                    patch_artist=True, widths=0.4,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#4c72b0')
    bp['boxes'][1].set_facecolor('#dd8452')

    for a, b in zip(ae_v, aae_v):
        ax.plot([1, 2], [a, b], color='gray', linewidth=0.8, alpha=0.6, zorder=3)
    ax.scatter([1]*len(ae_v),  ae_v,  color='#4c72b0', zorder=4, s=40)
    ax.scatter([2]*len(aae_v), aae_v, color='#dd8452', zorder=4, s=40)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel(title)

fig.suptitle(f'AE vs AAE — {len(df)} seeds, {N_EPOCHS} epochs each', fontsize=14)
fig.tight_layout()
plot_path = os.path.join(OUT_DIR, 'multi_run_summary.png')
fig.savefig(plot_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Plot saved to Drive: {plot_path}')

In [ ]:
# ── Cell 9: Final summary printout ───────────────────────────────────────────
print('FINAL RESULTS SUMMARY')
print('=' * 50)
for col_ae, col_aae, label in [
    ('ae_roc_auc', 'aae_roc_auc', 'ROC-AUC'),
    ('ae_pr_auc',  'aae_pr_auc',  'PR-AUC'),
    ('ae_f1_fpr',  'aae_f1_fpr',  'F1@FPR-5%'),
    ('ae_val_mse', 'aae_val_mse', 'Val MSE'),
]:
    ae_m  = df[col_ae].mean()
    ae_s  = df[col_ae].std()
    aae_m = df[col_aae].mean()
    aae_s = df[col_aae].std()
    print(f'{label:<12}  AE {ae_m:.4f} ± {ae_s:.4f}   AAE {aae_m:.4f} ± {aae_s:.4f}')
print('=' * 50)
print(f'Results directory: {OUT_DIR}')